Auditoria Gran Encuesta Integrada de Hogares 2026 DANE

4 MODULOS
- CGSE
- FT
- OCUPADOS
- DSEMPLEADOS

In [0]:
#Leyendo la tabla de caracteristicas generales
cgse = spark.read.format("delta").table("workspace.electiva.cgse")

In [0]:
#Leyendo la tabla de caracteristicas generales
cgse = spark.read.format("delta").table("workspace.electiva.cgse")
desempleados = spark.read.format("delta").table("workspace.electiva.desempleados")
ft = spark.read.format("delta").table("workspace.electiva.ft")
ocupados = spark.read.format("delta").table("workspace.electiva.ocupados")

# Convertir el DataFrame de Spark a Pandas
cgse = cgse.toPandas()
desempleados=desempleados.toPandas()
ft=ft.toPandas()
ocupados=ocupados.toPandas()

In [0]:
cgse

In [0]:
cgse['FEX_C18'].sum()

In [0]:
#Leyendo la tabla de fuerza de trabajo
ft = spark.read.format("delta").table("workspace.electiva.ft")
ft=ft.toPandas()
ft

Exploracion de modulos y calculos base

1.Poblacion total

In [0]:
cgse
#PT = cgse['FEX_C18'].sum()

In [0]:
D = desempleados[desempleados['DSI']==1]['FEX_C18'].sum()
D

In [0]:
D = desempleados[desempleados['DSI']==1]['FEX_C18'].sum()
D

segundo tema a resolver, como calcular PEA?

In [0]:
PEA = ft[ft['FT'] == 1]['FEX_C18'].sum()
PEA
## Cantidad de personas en la PEA en colombia 26593429.254050065

calculando la tasa de desempleo

In [0]:
TD = D/PEA*100
print(f'La tasa de desempleo en colombia para junio 2026 es de {TD} % ')

Actividad:
. calcular Tasa global de partipacion -> TGP = PEA/PET*100
· calcular tasa de ocupacion -> TO = O/PEA*100
O/PET*100

In [0]:
PEA = ft[ft['FT'] == 1]['FEX_C18'].sum()
PET = ft[ft['PET'] == 1]['FEX_C18'].sum()
TGP = (PEA/PET)* 100
print(TGP)

In [0]:
O = ocupados[ocupados['OCI'] == 1]['FEX_C18'].sum()
TO=O/PEA*100
TO=O/PET*100
print(TO)

In [0]:
#Calculando cuantas personas trabajan en el sector de alojamiento y servicios de comida en colombia a junio de 2026
ocupados[(ocupados['RAMA2D_R4']==55) | (ocupados['RAMA2D_R4']==56) ]['FEX_C18'].sum()
# En colombia en el sector de alojamiento y servicios de comida trabajan 2.057.354 personas a junio de 2026

In [0]:
#Calculando cuantas personas trabajan en el sector de construccion en colombia a junio de 2026
ocupados[(ocupados['RAMA2D_R4']==41) | (ocupados['RAMA2D_R4']==42) | (ocupados['RAMA2D_R4']==43) ]['FEX_C18'].sum()
# En colombia en el sector de construccion trabajan  1.678.378  personas a junio de 2026

In [0]:
#Calculando cuantas personas trabajan en Actividades financieras y de seguros  en colombia a junio de 2026
ocupados[(ocupados['RAMA2D_R4']==64) | (ocupados['RAMA2D_R4']==65) | (ocupados['RAMA2D_R4']==66) ]['FEX_C18'].sum()


Calculo de indicadores laborales para el departamento del Quindío Junio 2026 segmentado por genero.

Vamos a responder a la pregunta :

1.Cual es la tasa de desempleo en hombres en el quindio para junio 2026
2.Cual es la tasa de desempleo en mujeres en el quindio para junio 2026

In [0]:
desempleados

In [0]:
cgse[(cgse['DIRECTORIO']==8637743) & (cgse['SECUENCIA_P']==1) & (cgse['ORDEN']==1) ][['DIRECTORIO','SECUENCIA_P','ORDEN','P3271']]

Necesitamos traernos la variable p3271 del modulo de cgse, y añadirla al modulo de desempleados para poder calcular tasa de desempleo en hombres y mujeres del quindío

In [0]:
import pandas as pd
#Primero hay que anexar la p3271 a la tabla de desempleados
desempleados = desempleados.drop(columns=[c for c in desempleados.columns if c in ('P3271', 'P6040', 'P3271_x', 'P6040_x', 'P3271_y', 'P6040_y')])
desempleados = pd.merge(desempleados, cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271','P6040']], on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'], how='left')

In [0]:
desempleados

In [0]:
## Calcular cantidad de mujeres y hombres desempleados en el quindio

# 1.Cantidad de mujeres desempleadas en el quindio
MDQ=desempleados[(desempleados['P3271']==2) & (desempleados['DSI']==1) & (desempleados['DPTO']==63)]['FEX_C18'].sum()
HDQ=desempleados[(desempleados['P3271']==1) & (desempleados['DSI']==1) & (desempleados['DPTO']==63)]['FEX_C18'].sum()

print(f"en el quindio hay {MDQ} Mujeres desempleadas")
print(f"en el quindio hay {HDQ} Hombres desempleados")

In [0]:

## Primero hay que añadir la P3271 al modulo de fuerza de trabajo 
ft = pd.merge(ft, cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271', 'P6040']], on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'], how='left')

In [0]:
## Calcular PEA de mujeres y hombres en el quindio 

# 1.PEA de mujeres en el quindio
PEAMQ = ft[(ft['FT']==1) & (ft['DPTO']==63) & (ft['P3271']==2)]['FEX_C18'].sum()
# 2.PEA de hombres en el quindio
PEAHQ = ft[(ft['FT']==1) & (ft['DPTO']==63) & (ft['P3271']==1) ]['FEX_C18'].sum()

print(f"en el quindio hay {PEAMQ} Mujeres en la PEA")
print(f"en el quindio hay {PEAHQ} Hombres en la PEA")